In [1]:
# Cell 1: Load and Concatenate Normalized Training and Testing Datasets Across All Nodes

import os
import pandas as pd

# Define node identifiers and root directory for normalized datasets
NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
BASE_DIR = "dataset/normalized"

# Construct file paths for all node splits
train_files = [os.path.join(BASE_DIR, f"Node_{node}_train_normalized.csv") for node in NODES]
test_files = [os.path.join(BASE_DIR, f"Node_{node}_test_normalized.csv") for node in NODES]

# Load and combine training and testing datasets across all nodes
df_train = pd.concat([pd.read_csv(p) for p in train_files], ignore_index=True)
df_test = pd.concat([pd.read_csv(p) for p in test_files], ignore_index=True)

print("=== Combined Baseline Multi-Node Datasets (Nodes A–H) ===")
print(f"Combined Training Shape: {df_train.shape}")
print(f"Combined Testing Shape:  {df_test.shape}")

print("\nTarget Label Distribution (Train):")
print(df_train["Attack"].value_counts())

print("\nTarget Label Distribution (Test):")
print(df_test["Attack"].value_counts())

=== Combined Baseline Multi-Node Datasets (Nodes A–H) ===
Combined Training Shape: (448000, 7)
Combined Testing Shape:  (192000, 7)

Target Label Distribution (Train):
Attack
none         212100
syn-flood    110600
Backdoor     100800
noneX2        24500
Name: count, dtype: int64

Target Label Distribution (Test):
Attack
none         90900
syn-flood    47400
Backdoor     43200
noneX2       10500
Name: count, dtype: int64


In [2]:
# Cell 2: Feature Preprocessing, Model Training, and Evaluation

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, log_loss, mean_squared_error, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Define feature columns and expected ordering
TARGET_COL = "Attack"
NUMERIC_FEATURES = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
EXPECTED_FEATURE_ORDER = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]

# Extract features and targets
X_train = df_train[EXPECTED_FEATURE_ORDER].copy()
y_train_raw = df_train[TARGET_COL].copy()

X_test = df_test[EXPECTED_FEATURE_ORDER].copy()
y_test_raw = df_test[TARGET_COL].copy()

# Encode operational state: idle -> 0, charging -> 1
state_mapping = {"idle": 0, "charging": 1}
X_train["State"] = X_train["State"].map(state_mapping).fillna(0).astype(int)
X_test["State"] = X_test["State"].map(state_mapping).fillna(0).astype(int)

# Encode target categorical labels into integers
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

# Standardize numerical features using StandardScaler
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[NUMERIC_FEATURES] = scaler.fit_transform(X_train[NUMERIC_FEATURES])
X_test_scaled[NUMERIC_FEATURES] = scaler.transform(X_test[NUMERIC_FEATURES])

X_train_np = X_train_scaled.to_numpy()
X_test_np = X_test_scaled.to_numpy()

# Train Baseline Random Forest Classifier with OOB tracking
rf_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    min_samples_split=20,
    max_samples=0.8,
    random_state=42,
    oob_score=True,
    n_jobs=-1
)
rf_final.fit(X_train_np, y_train)

# Ensure predictions and probabilities are computed on final models
final_train_prob = rf_final.predict_proba(X_train_np)
final_test_prob = rf_final.predict_proba(X_test_np)

final_train_loss = log_loss(y_train, final_train_prob)
final_test_loss = log_loss(y_test, final_test_prob)

# Evaluate Out-Of-Bag (OOB) Performance
oob_probs = rf_final.oob_decision_function_
oob_preds = np.argmax(oob_probs, axis=1)

print("=== Out-Of-Bag (OOB) Performance ===")
print(f"  OOB Accuracy:          {rf_final.oob_score_:.6f}")
print(f"  Macro F1-Score:        {f1_score(y_train, oob_preds, average='macro'):.6f}")
print(f"  Mean Squared Error:    {mean_squared_error(y_train, oob_preds):.6f}")
print(f"  Log Loss:              {log_loss(y_train, oob_probs):.6f}")
print(f"  ROC-AUC Score:         {roc_auc_score(y_train, oob_probs, multi_class='ovr', average='macro'):.6f}")

# Evaluate Combined Test Set Performance
y_test_pred = rf_final.predict(X_test_np)
y_test_prob = rf_final.predict_proba(X_test_np)

print("\n=== Combined Test Set Performance ===")
print(f"  Macro F1-Score:        {f1_score(y_test, y_test_pred, average='macro'):.6f}")
print(f"  Mean Squared Error:    {mean_squared_error(y_test, y_test_pred):.6f}")
print(f"  Final Train Log Loss:  {final_train_loss:.6f}")
print(f"  Final Test Log Loss:   {final_test_loss:.6f}")
print(f"  ROC-AUC Score:         {roc_auc_score(y_test, y_test_prob, multi_class='ovr', average='macro'):.6f}")

=== Out-Of-Bag (OOB) Performance ===
  OOB Accuracy:          0.920243
  Macro F1-Score:        0.931252
  Mean Squared Error:    0.163783
  Log Loss:              0.202135
  ROC-AUC Score:         0.988385

=== Combined Test Set Performance ===
  Macro F1-Score:        0.931771
  Mean Squared Error:    0.160734
  Final Train Log Loss:  0.192623
  Final Test Log Loss:   0.201272
  ROC-AUC Score:         0.988528


In [3]:
# Cell 3: Serialize Model and Preprocessors to Disk

import os
import joblib

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(rf_final, os.path.join(MODEL_DIR, "rf_final.joblib"))
joblib.dump(le, os.path.join(MODEL_DIR, "label_encoder.joblib"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))

print(f"Successfully saved baseline model and preprocessors to '{MODEL_DIR}'")

Successfully saved baseline model and preprocessors to 'models'


In [4]:
# Cell 4: Granular Per-Node Evaluation on Test Splits

import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, log_loss, mean_squared_error, roc_auc_score

def prepare_node_features(df_raw, scaler_instance, expected_cols, numeric_cols):
    X = df_raw[expected_cols].copy()
    if X["State"].dtype == object or isinstance(X["State"].iloc[0], str):
        state_mapping = {"idle": 0, "charging": 1}
        X["State"] = X["State"].map(state_mapping).fillna(0).astype(int)
    X = X[expected_cols]
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler_instance.transform(X[numeric_cols])
    return X_scaled.to_numpy()

metrics_records = []

for node in NODES:
    test_path = os.path.join(BASE_DIR, f"Node_{node}_test_normalized.csv")
    df_test_node = pd.read_csv(test_path)

    X_test_node_np = prepare_node_features(df_test_node, scaler, EXPECTED_FEATURE_ORDER, NUMERIC_FEATURES)
    y_test_node = le.transform(df_test_node[TARGET_COL])

    y_test_node_pred = rf_final.predict(X_test_node_np)
    y_test_node_prob = rf_final.predict_proba(X_test_node_np)

    macro_f1 = f1_score(y_test_node, y_test_node_pred, average="macro")
    mse_error = mean_squared_error(y_test_node, y_test_node_pred)
    loss = log_loss(y_test_node, y_test_node_prob, labels=np.arange(len(le.classes_)))

    try:
        roc_auc = roc_auc_score(y_test_node, y_test_node_prob, multi_class="ovr", average="macro", labels=np.arange(len(le.classes_)))
    except ValueError:
        roc_auc = np.nan

    metrics_records.append({
        "Node": f"Node {node}",
        "Samples": len(df_test_node),
        "Macro F1": macro_f1,
        "MSE": mse_error,
        "Log Loss": loss,
        "ROC-AUC": roc_auc,
    })

df_metrics = pd.DataFrame(metrics_records)
print("=== Summary Across All Nodes (Baseline Model) ===")
print(df_metrics.to_string(index=False))

print("\n=== Macro Average Across All Nodes ===")
print(f"  Avg Macro F1-Score:    {df_metrics['Macro F1'].mean():.6f}")
print(f"  Avg Mean Squared Error:{df_metrics['MSE'].mean():.6f}")
print(f"  Avg Log Loss:          {df_metrics['Log Loss'].mean():.6f}")
print(f"  Avg ROC-AUC Score:     {df_metrics['ROC-AUC'].mean():.6f}")

/home/yannic/Desktop/Master_rbg/Semester2/Forschungsarbeit 2/.venvDatasets/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


=== Summary Across All Nodes (Baseline Model) ===
  Node  Samples  Macro F1      MSE  Log Loss  ROC-AUC
Node A    24000  0.902988 0.142542  0.215672      NaN
Node B    24000  0.926047 0.126917  0.189931 0.988031
Node C    24000  0.927189 0.164208  0.196988 0.987748
Node D    24000  0.932711 0.173542  0.223374 0.988326
Node E    24000  0.926467 0.198708  0.237347 0.988431
Node F    24000  0.934172 0.174750  0.193994 0.990668
Node G    24000  0.934218 0.184583  0.187554 0.990607
Node H    24000  0.913087 0.120625  0.165315 0.989826

=== Macro Average Across All Nodes ===
  Avg Macro F1-Score:    0.924610
  Avg Mean Squared Error:0.160734
  Avg Log Loss:          0.201272
  Avg ROC-AUC Score:     0.989091


In [5]:
paper = "Zur Etablierung eines lokalen Klassifikations-Baselines wurde ein Random-Forest-Modell auf den aggregierten Trainingsdaten aller Netzwerkknoten (A bis H) trainiert. Die kontinuierlichen elektrischen Merkmale wurden mittels eines Standard-Scalers z-transformiert, während die Betriebszustände (State) binär kodiert wurden. Das Modell wurde mit 100 Entscheidungsbäumen, einer maximalen Tiefe von 15 sowie Sub-Bagging (max_samples=0.8) parametriert, um Überanpassung zu minimieren und robuste Generalisierungseigenschaften sicherzustellen. Die Evaluation erfolgte sowohl anhand Out-Of-Bag (OOB)-Metriken als auch auf separaten, knotenspezifischen Testdatensätzen, womit eine verlässliche Baseline für die anschließende Untersuchung des kollaborativen Zielkonflikts geschaffen wurde. Für die Klassifikation wurde ein vorab per Hyperparameter-Tuning optimierter Random-Forest-Klassifikator geladen. Im Gegensatz zu Vorab-Modellen, die getrennt auf Original-, DAE- oder TabGAN-Daten liefen, nutzt dieses finale Modell den fusionierten Hybrid-Datensatz als robuste, realitätsnahe Baseline."

print(paper)

Zur Etablierung eines lokalen Klassifikations-Baselines wurde ein Random-Forest-Modell auf den aggregierten Trainingsdaten aller Netzwerkknoten (A bis H) trainiert. Die kontinuierlichen elektrischen Merkmale wurden mittels eines Standard-Scalers z-transformiert, während die Betriebszustände (State) binär kodiert wurden. Das Modell wurde mit 100 Entscheidungsbäumen, einer maximalen Tiefe von 15 sowie Sub-Bagging (max_samples=0.8) parametriert, um Überanpassung zu minimieren und robuste Generalisierungseigenschaften sicherzustellen. Die Evaluation erfolgte sowohl anhand Out-Of-Bag (OOB)-Metriken als auch auf separaten, knotenspezifischen Testdatensätzen, womit eine verlässliche Baseline für die anschließende Untersuchung des kollaborativen Zielkonflikts geschaffen wurde. Für die Klassifikation wurde ein vorab per Hyperparameter-Tuning optimierter Random-Forest-Klassifikator geladen. Im Gegensatz zu Vorab-Modellen, die getrennt auf Original-, DAE- oder TabGAN-Daten liefen, nutzt dieses fi